# MyNote: Summary

All this is doing is to create a view which selects from table data_science_db.public.customer_churn.


## ClaudeCode summary
 The "feature engineering" is just:
  
  - CAST on numeric columns
  - Simple arithmetic/boolean expressions (GENDER = 'Male' → 0/1, AGE >= 21 → 0/1)
  - Manual one-hot encoding (GEOGRAPHY = 'France' → 0/1, etc.)
  - array_construct() to pack those columns into an array

  All of which could be written as a straightforward CREATE VIEW ... AS SELECT in plain SQL. The Snowpark DataFrame API is doing nothing here that a simple SQL view couldn't do in one statement. It's a lab exercise that demonstrates the API mechanics rather than doing
  anything particularly sophisticated.

## Equivalent view created by ClaudeCode

NB I haven't tested this or tried to run the notebook

```sql
  CREATE OR REPLACE VIEW data_science_db.public.churn_features_vw AS
  SELECT
      CAST(CUSTOMER_ID AS INTEGER) AS id,
      CAST(CHURNED    AS INTEGER)  AS target,
      ARRAY_CONSTRUCT(
          CAST(TENURE                             AS FLOAT),  -- f1
          CAST(NUM_OF_PRODUCTS                    AS FLOAT),  -- f2
          CAST(HAS_AIRLINE_CREDIT_CARD            AS FLOAT),  -- f3
          CAST(IS_ACTIVE_MEMBER                   AS FLOAT),  -- f4
          CAST(ESTIMATED_SALARY                   AS FLOAT),  -- f5
          CASE WHEN GENDER    = 'Female'  THEN 1.0 ELSE 0.0 END,  -- f6
          CASE WHEN AGE       >= 21       THEN 1.0 ELSE 0.0 END,  -- f7
          CASE WHEN GEOGRAPHY = 'France'  THEN 1.0 ELSE 0.0 END,  -- f8
          CASE WHEN GEOGRAPHY = 'Spain'   THEN 1.0 ELSE 0.0 END,  -- f9
          CASE WHEN GEOGRAPHY = 'GERMANY' THEN 1.0 ELSE 0.0 END   -- f10
      ) AS features
  FROM data_science_db.public.customer_churn;
  ```
  
Example view output would apparently look like this for one row:

| id | target | features |
|---|---|---|
| 15634602 | 1 | [2.0, 1.0, 1.0, 0.0, 101348.88, 1.0, 1.0, 0.0, 0.0, 0.0] |


# Feature Engineering with a View

This notebook uses Snowpark DataFrame programming to produce a view that does feature engineering on the customer churn data. The finished transformation puts all features into a feature vector for machine learning.

### Steps below:

1. Connect to Snowflake
2. Access the churn table
3. Feature engineering
4. Save

In [ ]:
import snowflake.snowpark
from snowflake.snowpark.functions import *
from snowflake.snowpark.session import Session
from snowflake.snowpark.types import *

config_dir = '/home/jovyan/.ssh'
configfile = config_dir + '/sf_config'

### 1. Connect to Snowflake

In [ ]:
# Load configuration file
with open(configfile) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], 'rb') as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{'private_key': private_key_bytes}}).create()

### 2. Access the churn table

In [ ]:
churnDF = session.table('data_science_db.public.customer_churn')

### 3. Feature engineering

You can use the usual collection of DataFrame methods to transform a data set for feature engineering.

> There is one notable exception below: the **sql_expr()** function call near the end of the statement. 

>> This function lets you import any raw Snowflake SQL as a column expression into your DataFrame.

>> In this case, the statement uses the **array_construct()** function from Snowflake SQL to produce a vector (as an array of floating point numbers).

In [ ]:
churn_featuresDF = (
    churnDF  # See comments to the right >>>
    .withColumn('id', col('CUSTOMER_ID').cast(IntegerType()))             # Rename and cast columns
    .withColumn('target', col('CHURNED').cast(IntegerType()))
    .withColumn('f1', col('TENURE').cast(FloatType()))                                   
    .withColumn('f2', col('NUM_OF_PRODUCTS').cast(FloatType()))
    .withColumn('f3', col('HAS_AIRLINE_CREDIT_CARD').cast(FloatType()))
    .withColumn('f4', col('IS_ACTIVE_MEMBER').cast(FloatType()))
    .withColumn('f5', col('ESTIMATED_SALARY').cast(FloatType()))
    .withColumn('f6', col('GENDER')=='Female')                            # String encoder
    .withColumn('f7', col('AGE') >= 21)                                   # Binarizer
    .withColumn('f8', col('GEOGRAPHY')=='France')                         # One-
    .withColumn('f9', col('GEOGRAPHY')=='Spain')                          #   hot
    .withColumn('f10', col('GEOGRAPHY')=='GERMANY')                       #     encoding
    .withColumn('f6', col('f6').cast(IntegerType()).cast(FloatType()))    # Two steps to cast boolean to float
    .withColumn('f7', col('f7').cast(IntegerType()).cast(FloatType()))
    .withColumn('f8', col('f8').cast(IntegerType()).cast(FloatType()))
    .withColumn('f9', col('f9').cast(IntegerType()).cast(FloatType()))
    .withColumn('f10', col('f10').cast(IntegerType()).cast(FloatType()))
    .select('id', 'target', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10')
)

In [ ]:
# Assemble feature vector
churn_w_feature_vectorDF = (
    churn_featuresDF
    .withColumn('features', 
                sql_expr('array_construct(f1,f2,f3,f4,f5,f6,f7,f8,f9,f10)'))
    .select('id','target','features')
)                           

* Examine the DataFrame

In [ ]:
churn_w_feature_vectorDF.schema.fields

In [ ]:
churn_w_feature_vectorDF.count()

In [ ]:
churn_w_feature_vectorDF.show(2)

### 4. Save

In [ ]:
churn_w_feature_vectorDF.createOrReplaceView('public.churn_features_vw')